# Detecção de Fraude em Cartão de Crédito

**Desafio:** treinar modelos que detectam fraude em transações reais de cartão de crédito,
um problema em que a fraude é rara (~0,17% das transações) e a **acurácia engana**.

Este notebook segue o caminho: **Explorar → Preparar → Treinar → Avaliar → Explicar**.

> Antes de rodar: pegue o link do dataset na Aula 1 do desafio (é o dataset público
> *Credit Card Fraud Detection*, com as colunas `Time`, `Amount`, `Class` e `V1`...`V28`
> — variáveis já transformadas por PCA para proteger a privacidade).
> **Não suba o dataset para o repositório** — ele deve ser carregado pelo link/download,
> não versionado no GitHub.


## 0. Setup — bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score, f1_score
)

# XGBoost é opcional; se não estiver instalado, rode: pip install xgboost
from xgboost import XGBClassifier

import shap

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")


## 1. Explorar

Carregar a base, entender as colunas e medir a proporção de fraudes.


In [ ]:
# Opção A: carregar de um arquivo local baixado a partir do link da Aula 1
# df = pd.read_csv("creditcard.csv")

# Opção B: carregar direto de uma URL (ex.: se o link da Aula 1 for um CSV público)
DATA_URL = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"
df = pd.read_csv(DATA_URL)

df.head()


In [ ]:
df.info()
print("\nLinhas, colunas:", df.shape)


In [ ]:
# Proporção de fraude vs. transações normais
class_counts = df["Class"].value_counts()
class_pct = df["Class"].value_counts(normalize=True) * 100

print(class_counts)
print()
print(class_pct.round(4))

fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, ax=ax)
ax.set_xticklabels(["Normal (0)", "Fraude (1)"])
ax.set_ylabel("Número de transações")
ax.set_title("Desbalanceamento das classes")
plt.show()


**Por que a acurácia engana aqui:** um modelo que responde "não é fraude" para
TODAS as transações já acerta ~99,8% das vezes — e é completamente inútil, porque
erra 100% das fraudes (que é exatamente o que queremos detectar). Por isso o foco
das métricas vai para **recall, precisão e F1 da classe de fraude**, não a acurácia
geral.

In [ ]:
# Distribuição de Amount e Time
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["Amount"], bins=50, ax=axes[0])
axes[0].set_title("Distribuição de Amount")
sns.histplot(df["Time"], bins=50, ax=axes[1])
axes[1].set_title("Distribuição de Time (segundos desde a 1ª transação)")
plt.tight_layout()
plt.show()

df[["Time", "Amount"]].describe()


## 2. Preparar

- Criar `log(Amount)` (Amount tem cauda longa, log ajuda os modelos lineares)
- Padronizar as variáveis com `StandardScaler`
- Separar treino/teste com `stratify=y` para manter a proporção de fraudes nos dois


In [ ]:
df["log_amount"] = np.log1p(df["Amount"])

# Features: V1..V28 + log_amount + Time (Time normalmente ajuda pouco, mas deixamos
# para o modelo decidir a importância)
feature_cols = [c for c in df.columns if c.startswith("V")] + ["log_amount", "Time"]
X = df[feature_cols].copy()
y = df["Class"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Treino:", X_train.shape, " | Fraudes no treino:", y_train.sum(),
      f"({y_train.mean()*100:.3f}%)")
print("Teste: ", X_test.shape, " | Fraudes no teste: ", y_test.sum(),
      f"({y_test.mean()*100:.3f}%)")


In [ ]:
# Padronização — ajustar (fit) só no treino, aplicar (transform) em treino e teste,
# para não vazar informação do teste
scaler = StandardScaler()
cols_to_scale = ["log_amount", "Time"]  # V1..V28 já vêm padronizadas pelo PCA original

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])


**Undersampling vs. oversampling:** a regra da Expert é testar. Deixe abaixo pronto
para comparar (opcional — comece sem isso, usando `class_weight`/`scale_pos_weight`,
e só depois compare).

In [ ]:
# OPCIONAL — comparar estratégias de balanceamento
# from imblearn.under_sampling import RandomUnderSampler
# from imblearn.over_sampling import SMOTE
#
# rus = RandomUnderSampler(random_state=RANDOM_STATE)
# X_train_under, y_train_under = rus.fit_resample(X_train_scaled, y_train)
#
# smote = SMOTE(random_state=RANDOM_STATE)
# X_train_over, y_train_over = smote.fit_resample(X_train_scaled, y_train)
#
# Treine os mesmos modelos abaixo com X_train_under/y_train_under e X_train_over/y_train_over
# e compare o recall de fraude de cada estratégia (under x over x class_weight).


## 3. Treinar

Baseline: regressão logística. Depois comparar com Random Forest e XGBoost, usando
peso das classes e, mais à frente, ajuste do limiar de decisão.


In [ ]:
# Baseline: Regressão Logística
log_reg = LogisticRegression(
    class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE
)
log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)
y_proba_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred_lr, digits=4))


In [ ]:
# Random Forest
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train_scaled, y_train)

y_pred_rf = rf.predict(X_test_scaled)
y_proba_rf = rf.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred_rf, digits=4))


In [ ]:
# XGBoost — scale_pos_weight compensa o desbalanceamento (proporção neg/pos)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb.fit(X_train_scaled, y_train)

y_pred_xgb = xgb.predict(X_test_scaled)
y_proba_xgb = xgb.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred_xgb, digits=4))


In [ ]:
# OPCIONAL — ampliar a busca de hiperparâmetros com GridSearchCV (pode demorar)
# from sklearn.model_selection import StratifiedKFold
#
# param_grid = {
#     "n_estimators": [200, 400],
#     "max_depth": [4, 6, 8],
#     "learning_rate": [0.05, 0.1],
# }
# cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
# grid = GridSearchCV(
#     XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric="aucpr",
#                    random_state=RANDOM_STATE),
#     param_grid, scoring="average_precision", cv=cv, n_jobs=-1
# )
# grid.fit(X_train_scaled, y_train)
# print(grid.best_params_, grid.best_score_)


## 4. Avaliar

Classification report, curva ROC e curva precisão-recall, com o **recall da fraude**
como métrica principal.


In [ ]:
models_proba = {
    "Logistic Regression": y_proba_lr,
    "Random Forest": y_proba_rf,
    "XGBoost": y_proba_xgb,
}

# Comparação em tabela: recall, precisão e F1 da classe fraude (limiar padrão 0.5)
rows = []
preds = {"Logistic Regression": y_pred_lr, "Random Forest": y_pred_rf, "XGBoost": y_pred_xgb}
for name, y_pred in preds.items():
    from sklearn.metrics import precision_score, recall_score
    rows.append({
        "modelo": name,
        "recall_fraude": recall_score(y_test, y_pred),
        "precisao_fraude": precision_score(y_test, y_pred),
        "f1_fraude": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, models_proba[name]),
        "avg_precision": average_precision_score(y_test, models_proba[name]),
    })

# Ordenamos por F1 (equilíbrio entre recall e precisão), não só por recall,
# para escolher um "melhor modelo" consistente com o resto do notebook
# (a etapa de explicação com SHAP também usa esse modelo).
comparison_df = pd.DataFrame(rows).sort_values("f1_fraude", ascending=False)
comparison_df


In [ ]:
# Curva ROC
fig, ax = plt.subplots(figsize=(6, 5))
for name, proba in models_proba.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.4f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Curva ROC")
ax.legend()
plt.show()


In [ ]:
# Curva Precisão-Recall — mais informativa que ROC quando as classes são tão desbalanceadas
fig, ax = plt.subplots(figsize=(6, 5))
for name, proba in models_proba.items():
    precision, recall, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    ax.plot(recall, precision, label=f"{name} (AP={ap:.4f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precisão")
ax.set_title("Curva Precisão-Recall (classe fraude)")
ax.legend()
plt.show()


### Ajuste do limiar de decisão

O limiar padrão de 0.5 raramente é o melhor em problemas desbalanceados. Aqui
testamos vários limiares no melhor modelo (o de maior F1 na tabela acima) e
escolhemos o limiar que maximiza o F1 da fraude — ajuste esse critério conforme
a prioridade do negócio (ex.: se falso negativo é muito mais caro que falso
positivo, priorize recall em vez de F1).


In [ ]:
best_model_name = comparison_df.iloc[0]["modelo"]
best_proba = models_proba[best_model_name]
print("Melhor modelo (por F1):", best_model_name)

thresholds = np.arange(0.05, 0.95, 0.05)
threshold_rows = []
for t in thresholds:
    y_pred_t = (best_proba >= t).astype(int)
    threshold_rows.append({
        "limiar": round(t, 2),
        "recall": recall_score(y_test, y_pred_t),
        "precisao": precision_score(y_test, y_pred_t),
        "f1": f1_score(y_test, y_pred_t),
    })

threshold_df = pd.DataFrame(threshold_rows)
best_threshold_row = threshold_df.loc[threshold_df["f1"].idxmax()]
print(best_threshold_row)

threshold_df.plot(x="limiar", y=["recall", "precisao", "f1"], figsize=(7, 4),
                   title=f"Recall/Precisão/F1 por limiar — {best_model_name}")
plt.show()


In [ ]:
CHOSEN_THRESHOLD = best_threshold_row["limiar"]  # ajuste manualmente se preferir outro
y_pred_final = (best_proba >= CHOSEN_THRESHOLD).astype(int)

print(f"Limiar escolhido: {CHOSEN_THRESHOLD}")
print(classification_report(y_test, y_pred_final, digits=4))

cm = confusion_matrix(y_test, y_pred_final)
fig, ax = plt.subplots(figsize=(4, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal", "Fraude"], yticklabels=["Normal", "Fraude"], ax=ax)
ax.set_xlabel("Previsto")
ax.set_ylabel("Real")
ax.set_title(f"Matriz de confusão — {best_model_name} (limiar={CHOSEN_THRESHOLD})")
plt.show()


## 5. Explicar

Importância das variáveis e SHAP para mostrar o que pesou em cada decisão do modelo.


In [ ]:
# Importância das variáveis (nativa do modelo de árvore escolhido, ex.: XGBoost)
model_for_explanation = xgb  # troque para rf se o Random Forest tiver sido o melhor

importances = pd.Series(
    model_for_explanation.feature_importances_, index=X_train_scaled.columns
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 8))
importances.head(15).sort_values().plot(kind="barh", ax=ax)
ax.set_title("Top 15 variáveis mais importantes")
plt.tight_layout()
plt.show()


In [ ]:
# SHAP — explica a contribuição de cada variável para cada previsão
explainer = shap.TreeExplainer(model_for_explanation)

# Para acelerar, explique uma amostra do teste (ex.: 1000 linhas)
sample = X_test_scaled.sample(n=min(1000, len(X_test_scaled)), random_state=RANDOM_STATE)
shap_values = explainer.shap_values(sample)

shap.summary_plot(shap_values, sample, show=True)


In [ ]:
# Explicação de um caso específico: uma fraude verdadeira detectada pelo modelo
fraud_idx_in_test = y_test[y_test == 1].index
if len(fraud_idx_in_test) > 0:
    example_idx = fraud_idx_in_test[0]
    example_row = X_test_scaled.loc[[example_idx]]
    example_shap = explainer.shap_values(example_row)

    shap.force_plot(
        explainer.expected_value, example_shap[0], example_row.iloc[0],
        matplotlib=True, show=True
    )


## 6. O que mudei em relação ao caminho da Expert

- **Critério de escolha do "melhor modelo":** em vez de escolher pelo recall
  isoladamente (o que levaria à Regressão Logística, que detecta 92% das
  fraudes mas erra em 94% dos alertas — precisão de só 6%), escolhi pelo
  **F1-score da classe fraude**, que equilibra recall e precisão. Isso trocou
  o modelo escolhido para o **XGBoost**.
- **Ajuste do limiar de decisão:** testei limiares de 0.05 a 0.90 no XGBoost e
  o que maximizou o F1 foi **0.75** (acima do padrão de 0.5), resultando em
  **recall de 82,7%, precisão de 90% e F1 de 0,86** na classe fraude — ou
  seja, o modelo detecta 5 em cada 6 fraudes, com 9 em cada 10 alertas sendo
  fraude de verdade.
- **Trade-off assumido:** optei por não maximizar o recall isoladamente,
  porque um modelo com recall altíssimo e precisão baixa (como a Regressão
  Logística neste caso) geraria um volume grande demais de falsos alertas
  para uma equipe investigar na prática. O F1 como critério busca um
  equilíbrio mais realista para uso operacional.
